Listing of All Businesses
https://data.lacity.org/Administration-Finance/Listing-of-All-Businesses/r4uk-afju/about_data


### Visualization

In [5]:
import pandas as pd
df_cleaned = pd.read_csv('dataset\\business_filtered.csv')

In [6]:
df_cleaned.shape

(515273, 24)

In [ ]:
# df_cleaned1['NAICS-2'] = df_cleaned1['NAICS'].map(lambda n: int(n / 10000))
# df_cleaned1.to_csv('dataset\\business_subset.csv', index=False)

In [30]:
import pandas as pd
url = 'https://raw.githubusercontent.com/EricSJSU-DataScience/CS163_project/refs/heads/main/dataset/business_subset.csv'
file_path = 'dataset\\business_subset.csv'
df_map = pd.read_csv(url)

In [32]:
df_map.head(5)

,BUSINESS NAME,STREET ADDRESS,NAICS,latitude,longitude,5d_zip,NAICS-2
0,MARIA OFELIA NUNEZ,1017 W 88TH STREET,444130,33.9581,-118.2926,90044,44
1,SKAB INC,21731 VENTURA BLVD SUITE #300,532290,34.1692,-118.6016,91364,53
2,PRODYNAMICS INC,12099 W WASHINGTON UNIT #408,621340,33.9982,-118.4238,90066,62
3,LIZARRAGA TRANSPORT INC,9663 CARRON DRIVE,488000,33.9803,-118.0841,90660,48
4,MARGARIT KARAPETIAN,16425 BUCHET DRIVE,541219,34.2879,-118.4900,91344,54


In [25]:
naics_counts = df_map['NAICS-2'].value_counts().to_dict()
naics_options = [{"label": f"{code} - {code_sector_dict.get(code, 'Unknown')} ({naics_counts.get(code, 0)})", "value": code}
                  for code in sorted(naics_counts.keys())]
naics_options

[{'label': '11 - Agriculture, Forestry, Fishing and Hunting (1184)',
  'value': 11},
 {'label': '21 - Mining (95)', 'value': 21},
 {'label': '22 - Utilities (6)', 'value': 22},
 {'label': '23 - Construction (38060)', 'value': 23},
 {'label': '31 - Manufacturing (12233)', 'value': 31},
 {'label': '32 - Manufacturing (2155)', 'value': 32},
 {'label': '33 - Manufacturing (6310)', 'value': 33},
 {'label': '42 - Wholesale Trade (31694)', 'value': 42},
 {'label': '44 - Retail Trade (30190)', 'value': 44},
 {'label': '45 - Retail Trade (28703)', 'value': 45},
 {'label': '48 - Transportation and Warehousing (5015)', 'value': 48},
 {'label': '49 - Transportation and Warehousing (3745)', 'value': 49},
 {'label': '51 - Information (20579)', 'value': 51},
 {'label': '52 - Finance and Insurance (7347)', 'value': 52},
 {'label': '53 - Real Estate Rental and Leasing (61513)', 'value': 53},
 {'label': '54 - Professional, Scientific, and Technical Services (66525)',
  'value': 54},
 {'label': '55 - Man

In [23]:
df_naics = pd.read_csv('dataset\\naics_2_clean.csv')
code_sector_dict = df_naics.set_index('Code')['Sector_Title'].to_dict()
code_sector_dict

{11: 'Agriculture, Forestry, Fishing and Hunting',
 21: 'Mining',
 22: 'Utilities',
 23: 'Construction',
 31: 'Manufacturing',
 32: 'Manufacturing',
 33: 'Manufacturing',
 42: 'Wholesale Trade',
 44: 'Retail Trade',
 45: 'Retail Trade',
 48: 'Transportation and Warehousing',
 49: 'Transportation and Warehousing',
 51: 'Information',
 52: 'Finance and Insurance',
 53: 'Real Estate Rental and Leasing',
 54: 'Professional, Scientific, and Technical Services',
 55: 'Management of Companies and Enterprises',
 56: 'Administrative and Support and Waste… Services',
 61: 'Educational Services',
 62: 'Health Care and Social Assistance',
 71: 'Arts, Entertainment, and Recreation',
 72: 'Accommodation and Food Services',
 81: 'Other Services (except Public Administration)',
 92: 'Public Administration'}

In [36]:
df_cleaned['NAICS_2'] = df_cleaned['NAICS'].astype(str).str[:2]
df_cleaned['NAICS_2'] = df_cleaned['NAICS_2'].astype(int)

In [37]:
df_cleaned['NAICS_2_Title'] = df_cleaned['NAICS_2'].map(code_sector_dict)

In [38]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515273 entries, 0 to 515272
Data columns (total 26 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   LOCATION ACCOUNT #         515273 non-null  object 
 1   BUSINESS NAME              515273 non-null  object 
 2   DBA NAME                   173707 non-null  object 
 3   STREET ADDRESS             515273 non-null  object 
 4   CITY                       515271 non-null  object 
 5   ZIP CODE                   515273 non-null  object 
 6   LOCATION DESCRIPTION       515270 non-null  object 
 7   MAILING ADDRESS            232534 non-null  object 
 8   MAILING CITY               232545 non-null  object 
 9   MAILING ZIP CODE           232478 non-null  object 
 10  NAICS                      515273 non-null  int64  
 11  PRIMARY NAICS DESCRIPTION  515273 non-null  object 
 12  COUNCIL DISTRICT           515273 non-null  int64  
 13  LOCATION START DATE        51

In [56]:
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

# Example data:
# df has columns: 'duration', 'is_open', 'industry'
# is_open == 'No' => event occurred (closed), else censored
# industry => group label
def industry_logrank_test(df, industry_A, industry_B):
    # Suppose you want to compare survival in two industries, A and B:
    df_A = df[(df['NAICS_2_Title'] == industry_A)]
    df_B = df[(df['NAICS_2_Title'] == industry_B)]

    # duration: time to event or censor
    # event_observed: 1 if closed, 0 if still open
    results = logrank_test(
        durations_A = df_A['duration'],
        durations_B = df_B['duration'],
        event_observed_A = (df_A['is_open'] == 'No').astype(int),
        event_observed_B = (df_B['is_open'] == 'No').astype(int)
    )

    return results.print_summary()


In [57]:
industry_A = industry_list[0]
industry_B = industry_list[2]
print(f'{industry_A} vs {industry_B}')
industry_logrank_test(df_cleaned, industry_A, industry_B)

Retail Trade vs Professional, Scientific, and Technical Services


<lifelines.StatisticalResult: logrank_test>
               t_0 = -1
 null_distribution = chi squared
degrees_of_freedom = 1
         test_name = logrank_test

---
 test_statistic      p  -log2(p)
        1884.07 <0.005       inf

In [58]:
results = multivariate_logrank_test(df_cleaned['duration'], df_cleaned['NAICS_2_Title'], df_cleaned['is_open'])
results.print_summary()

<lifelines.StatisticalResult: multivariate_logrank_test>
               t_0 = -1
 null_distribution = chi squared
degrees_of_freedom = 20
         test_name = multivariate_logrank_test

---
 test_statistic      p  -log2(p)
       36016.10 <0.005       inf

In [41]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 515273 entries, 0 to 515272
Data columns (total 24 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   LOCATION ACCOUNT #         515273 non-null  object 
 1   BUSINESS NAME              515273 non-null  object 
 2   DBA NAME                   173707 non-null  object 
 3   STREET ADDRESS             515273 non-null  object 
 4   CITY                       515271 non-null  object 
 5   ZIP CODE                   515273 non-null  object 
 6   LOCATION DESCRIPTION       515270 non-null  object 
 7   MAILING ADDRESS            232534 non-null  object 
 8   MAILING CITY               232545 non-null  object 
 9   MAILING ZIP CODE           232478 non-null  object 
 10  NAICS                      515273 non-null  int64  
 11  PRIMARY NAICS DESCRIPTION  515273 non-null  object 
 12  COUNCIL DISTRICT           515273 non-null  int64  
 13  LOCATION START DATE        51

### Interactive survival curve

In [38]:
import pandas as pd
url = 'https://media.githubusercontent.com/media/EricSJSU-DataScience/CS163_project/refs/heads/main/dataset/business_filtered.csv'
file_path = 'dataset\\business_filtered.csv'
df = pd.read_csv(file_path, usecols=["NAICS", "is_open", "duration"])

In [39]:
df['is_open'].unique()

array([ True, False])

In [40]:
df['NAICS-2'] = df['NAICS'].map(lambda n: int(n / 10000))

In [41]:
df_naics = pd.read_csv('https://raw.githubusercontent.com/EricSJSU-DataScience/CS163_project/refs/heads/main/dataset/naics_2_clean.csv')
code_sector_dict = df_naics.set_index('Code')['Sector_Title'].to_dict()

In [42]:
df['NAICS-2_Title'] = df['NAICS-2'].map(code_sector_dict)

In [43]:
df.to_csv('dataset\\business_survivalplot.csv', index=False)

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def plot_kaplan_meier_by_industries(df, industries, max_time=600):
    """
    Computes and plots Kaplan-Meier survival curves for closed businesses
    for multiple industries using Plotly.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame that must include:
         - 'is_open': Where closed businesses are marked as 'No'.
         - 'duration': The duration (in months) of each business.
         - 'NAICS_2_Title': The industry classification.
    industries : list
        List of industry names to filter on (e.g., values from df['NAICS_2_Title'].unique()).
    max_time : int, optional
        Maximum time (in months) to display on the x-axis (default is 600).
    """
    
    fig = go.Figure()
    
    # Loop over each industry in the provided list.
    for industry in industries:
        # Filter for closed businesses in the given industry.
        if pd.isna(industry):
            df_ind = df[(df['NAICS-2_Title'].isna())].copy()
            industry_label = "NaN"
        else:
            df_ind = df[(df['NAICS-2_Title'] == industry)].copy()
            industry_label = industry
        
        # Skip if no data is available for this industry.
        if df_ind.empty:
            continue
        
        # Round the duration values.
        df_ind['duration_rounded'] = df_ind['duration'].round(1)
        
        # Count the number of closures (events) at each unique rounded duration.
        event_counts = df_ind[df_ind["is_open"] == False]["duration_rounded"].value_counts().sort_index()
        unique_times = event_counts.index

        # Compute the number at risk just before each event time.
        n_at_risk = [ (df_ind['duration_rounded'] >= t).sum() for t in unique_times ]
        
        # Compute the Kaplan-Meier survival probabilities.
        survival_probs = []
        cum_survival = 1.0
        for t, d, n in zip(unique_times, event_counts, n_at_risk):
            cum_survival *= (1 - d/n)
            survival_probs.append(cum_survival)
        
        # Add a step-like trace for the current industry.
        fig.add_trace(go.Scatter(
            x=list(unique_times),
            y=survival_probs,
            mode='lines',
            line_shape='hv',  # horizontal-vertical step plot
            name=str(industry_label)
        ))
    
    # Define x-axis ticks (every 12 months).
    xticks = list(np.arange(0, max_time + 12, 12))
    
    # Update the layout.
    fig.update_layout(
        title="Kaplan-Meier Survival Curves by Industry",
        xaxis_title="Duration (Months)",
        yaxis_title="Survival Probability",
        xaxis=dict(range=[0, max_time], tickmode='array', tickvals=xticks, tickangle=-90),
        yaxis=dict(range=[0, 1], tickmode='linear', dtick=0.1),
        legend=dict(title='Industry', font=dict(size=10), xanchor='right', yanchor='top'),
        width=1000, 
        height=600
    )
    
    fig.show()

# Example usage:
# industries_list = ['Retail Trade', 'Health Care and Social Assistance', 'Manufacturing']
# plot_kaplan_meier_by_industries(df_retail, industries_list)


In [8]:
industry_list = [
       'Retail Trade', 
       'Transportation and Warehousing',
       'Professional, Scientific, and Technical Services',
       'Arts, Entertainment, and Recreation', 'Information',
       'Construction', 
       'Wholesale Trade', 'Accommodation and Food Services',
       'Manufacturing', 'Finance and Insurance',
       'Educational Services'
       ]

In [ ]:
industry_options = [
    {"label": industry, "value": industry} for industry in industry_list
]

In [23]:
industry_options = [
    {"label": industry, "value": industry} 
    for industry in df['NAICS-2_Title'].unique() if pd.notnull(industry)
]

In [61]:
available_industries = sorted(df['NAICS-2_Title'].dropna().unique())

In [59]:
plot_kaplan_meier_by_industries(df, industry_list, 360)

### dataset modify 

In [26]:
url = 'https://media.githubusercontent.com/media/EricSJSU-DataScience/CS163_project/refs/heads/main/dataset/business_filtered.csv'
file_path = 'dataset\\business_filtered.csv'
df = pd.read_csv(file_path)

In [28]:
df['is_open'] = df['is_open'].map({'Yes': True, 'No': False})

In [30]:
df.to_csv('dataset\\business_filtered.csv', index=False)